# Brain MRI (magnetic resonance imaging) segmentation


## The task

This project addresses the segmentation of FLAIR abnormalities in brain MR images. Each image in the dataset is paired with a manually created FLAIR abnormality segmentation mask, so the goal is to segment the abnormal (tumor) region in lower-grade glioma cases.

Source kaggle challenge: https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation/data

## Notebook structure:
- 1. Dataset





In [ ]:
import kagglehub
from glob import glob
import cv2
import numpy as np
import pandas as pd
import seaborn as sns


In [ ]:
TRAIN_SPLIT = 0.7
TEST_SPLIT = 0.5

RANDOM_STATE = 42

## **1. The dataset**

The **LGG Segmentation Dataset** contains brain MR images together with manual FLAIR abnormality segmentation masks. The images were obtained from *The Cancer Imaging Archive* (TCIA) and correspond to **110 patients** included in *The Cancer Genome Atlas* (TCGA) lower-grade glioma collection, each having at least a fluid-attenuated inversion recovery (FLAIR) sequence and genomic cluster data available.

Tumor genomic clusters and patient data are provided in the `data.csv` file. For more information on the genomic data, refer to the publication *"Comprehensive, Integrative Genomic Analysis of Diffuse Lower-Grade Gliomas"* and its supplementary material: https://www.nejm.org/doi/full/10.1056/NEJMoa1402121

In [ ]:
# Download latest version
path = kagglehub.dataset_download("mateuszbuda/lgg-mri-segmentation")

print("Path to dataset files:", path)

In [ ]:
mask_paths = glob("/kaggle/input/**/*mask*.*", recursive=True)

print("Number of masks:", len(mask_paths))

for i,path in enumerate(mask_paths[:3]):
    print(f"{i}° path: {path}")

Every directory inside the dataset contains pairs of images:
- name_img.tif
- name_img_mask.tif

In [ ]:
images_paths = []

for i in masks_paths:
    images_paths.append(i.replace('_mask', ''))

df = pd.DataFrame(data= {'images_paths': images_paths, 'masks_paths': masks_paths})

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# create training dataframe
train_df, val_test_df = train_test_split(df, random_state = RANDOM_STATE, train_size= TRAINING_SPLIT)

# create validation and test dataframe
val_df, test_df = train_test_split(val_test_df, train_size= TEST_SPLIT)


## **2. Prepare data**

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class AugConfig:
    rotation_range: float = 0.0
    width_shift_range: float = 0.0
    height_shift_range: float = 0.0
    shear_range: float = 0.0
    zoom_range: float = 0.0
    horizontal_flip: bool = False
    fill_mode: str = 'nearest'

In [ ]:
def get_generator(df, aug_config, img_size, batch_size):
    
    params = asdict(aug_config)
    img_gen = ImageDataGenerator(**params)
    msk_gen = ImageDataGenerator(**params)

    # Create general generator
    img_gen_df = img_gen.flow_from_dataframe(df, x_col='images_paths', 
                                             class_mode=None, 
                                             color_mode='rgb', 
                                             target_size=img_size,
                                             batch_size=batch_size, 
                                             save_to_dir=None, 
                                             save_prefix='image', 
                                             seed=1)

    msk_gen_df = msk_gen.flow_from_dataframe(df, 
                                             x_col='masks_paths', 
                                             class_mode=None, 
                                             color_mode='grayscale', 
                                             target_size=img_size,
                                             batch_size=batch_size, 
                                             save_to_dir=None, 
                                             save_prefix= 'mask', 
                                             seed=1)

    gen = zip(img_gen_df, msk_gen_df)

    for (img, msk) in gen:
        img = img / 255
        msk = msk / 255
        msk[msk > 0.5] = 1
        msk[msk <= 0.5] = 0

        yield (img, msk)

In [ ]:
img_size = (256,256)
batch_size = 40

train_aug = AugConfig(rotation_range=0.2, 
                      width_shift_range=0.05, 
                      height_shift_range=0.05,
                      shear_range=0.05,
                      zoom_range=0.05, 
                      horizontal_flip=True,
                      fill_mode='nearest')


valid_aug = AugConfig()
test_aug = AugConfig()

train_gen = get_generator(train_df, train_aug, img_size, batch_size)
valid_gen = get_generator(valid_df, valid_aug, img_size, batch_size)
test_gen = get_generator(test_df, test_aug, img_size, batch_size)


## References

This dataset is used in:

- Mateusz Buda, Ashirbani Saha, Maciej A. Mazurowski. *"Association of genomic subtypes of lower-grade gliomas with shape features automatically extracted by a deep learning algorithm."* Computers in Biology and Medicine, 2019.
- Maciej A. Mazurowski, Kal Clark, Nicholas M. Czarnek, Parisa Shamsesfandabadi, Katherine B. Peters, Ashirbani Saha. *"Radiogenomics of lower-grade glioma: algorithmically-assessed tumor shape is associated with tumor genomic subtypes and patient outcomes in a multi-institutional study with The Cancer Genome Atlas data."* Journal of Neuro-Oncology, 2017.